# 🚀 Lab 1.3 — Databricks Workspace Setup from Scratch

## Learning Objectives
In this notebook, you will learn:
1. **MLflow 3 Landscape** — See the full evaluation map — three paradigms, eight modules, one `mlflow.genai` namespace
2. **Package Setup** — Install `mlflow[databricks]>=3.1` and `databricks-openai` in a Databricks notebook
3. **Unity Catalog** — Create a catalog, schema, and volume to hold every tutorial asset
4. **MLflow Experiment** — Register a workspace experiment so traces and runs land in a known location
5. **Foundation Model API** — Verify access to `databricks-claude-opus-4-6` via the OpenAI-compatible client
6. **Sanity-Check Trace** — Use `mlflow.openai.autolog()` to confirm tracing is wired end-to-end

## Prerequisites
- Databricks workspace with Foundation Model APIs enabled
- DBR 15.4 ML LTS (or newer) cluster with internet egress
- Permission to create Unity Catalog catalogs/schemas (or an existing catalog you can use)


---
## The MLflow 3 Evaluation Landscape

Before we configure anything, here is the full picture of what you'll build over the rest of the tutorial. Every later module slots into one of these layers — keep this map in mind as you go.

### Three Evaluation Paradigms (You'll Use All Three)

1. **Code-based scorers** — deterministic checks: regex, length bounds, JSON validity, custom Python predicates. Cheap, instant, perfect for hard rules ("must contain a citation", "must be ≤200 words").
2. **LLM-as-Judge** — a calibrated LLM grades outputs against rubrics or `expected_facts`. Scales semantic quality assessment at a fraction of human cost.
3. **Human feedback** — domain experts label a sample of outputs. Slow and expensive, but the only ground truth for ambiguous or high-stakes cases. Used to *calibrate* the LLM judges.

Production stacks **layer all three**: code scorers catch hard violations, LLM judges grade quality at scale, humans validate a sample and correct judge drift.

### The Pieces You'll Build (Module → Capability Map)

| Layer | Module | What it gives you | Key API |
| --- | --- | --- | --- |
| **Setup & Tracing** | 1 *(this module)* | Workspace, experiment, UC namespace, FM API access, traces flowing | `mlflow.openai.autolog()` |
| **Eval datasets** | 2 | Versioned UC-backed datasets from hand/traces/synthetic sources | `mlflow.genai.datasets.create_dataset` |
| **`predict_fn` patterns** | 3 | One `evaluate()` call works against local fns, endpoints, registered models, async | `mlflow.genai.evaluate` |
| **Built-in judges** | 4 | `Correctness`, `RelevanceToQuery`, `RetrievalGroundedness`, `Safety`, `Guidelines` | `mlflow.genai.scorers` |
| **Custom judges** | 5 | Domain-specific scorers tuned to your rubrics and calibrated against humans | `@mlflow.genai.scorer` |
| **Human review** | 6 | Labeling sessions that produce ground truth and improve judges | Review App / labeling sessions |
| **CI/CD gates** | 7 | Block deploys that regress on eval scores | `evaluate()` in pipelines |
| **Production monitoring** | 8 | Continuous scoring on live traffic; alerts on drift | Inference tables + scheduled jobs |

### Why MLflow 3 (and the `mlflow.genai` Namespace)

MLflow 3 unified tracing, evaluation, judges, datasets, and serving artifacts under a single `mlflow.genai` namespace. Earlier versions had fragmented APIs across `mlflow.evaluate`, `mlflow.metrics.genai`, and provider-specific integrations — these still exist but are superseded. **Everything in this tutorial uses the `mlflow.genai` API.**

### What This Lab Sets Up (Foundation for All Eight Modules)

By the end of this notebook, the four runtime pre-requisites every later lab depends on will be in place:

1. **Packages** — `mlflow[databricks]>=3.1` + `databricks-openai` installed
2. **UC namespace** — `genai_eval_tutorial.module_01` catalog/schema + a managed volume for unstructured assets
3. **Experiment** — `/Users/<you>/genai-eval-tutorial`, the home for every trace and eval run that follows
4. **Foundation Model API access** — verified call to `databricks-claude-opus-4-6` (the model we'll use as both agent and judge)
5. **Tracing verified end-to-end** — `mlflow.openai.autolog()` confirmed to write a trace visible in the UI


---
## Step 1 — Install Packages

We pin `mlflow[databricks]` to `>=3.1` because GenAI evaluation, tracing, and the new judge framework all require MLflow 3.x. `databricks-openai` gives us a drop-in OpenAI-compatible client that authenticates against your workspace automatically.


In [1]:
# ============================================================================
# 📦 STEP 1 - INSTALL PACKAGES
# ============================================================================

%pip install --quiet "mlflow[databricks]>=3.1" databricks-openai

dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.

Error: The "path" argument must be of type string. Received type undefined

---
## Step 2 — Configure Unity Catalog

We create a dedicated catalog and schema so every artifact produced in later labs (eval datasets, judge outputs, traces tables) lives in a predictable location. Replace the defaults below if your workspace requires specific naming.


In [2]:
# ============================================================================
# 🗂️ SET TUTORIAL NAMESPACE
# ============================================================================

CATALOG  = "genai_eval_tutorial"
SCHEMA   = "module_01"
VOLUME   = "assets"

print(f"Target namespace: {CATALOG}.{SCHEMA}")

Target namespace: genai_eval_tutorial.module_01

In [3]:
# ============================================================================
# 🗂️ CREATE CATALOG, SCHEMA, AND A MANAGED VOLUME
# ============================================================================

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA  {SCHEMA}")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))


databaseName
default
information_schema
module_01


---
## Step 3 — Set the MLflow Experiment

Every trace, evaluation run, and judge call needs an **experiment** to land in. We use a workspace path under your user folder so it shows up in the MLflow UI sidebar.


In [4]:
# ============================================================================
# 🧪 STEP 3 - SET THE MLFLOW EXPERIMENT
# ============================================================================

import mlflow

# Resolve the current user so the experiment path is unique per learner
USER_EMAIL = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)

EXPERIMENT_PATH = f"/Users/{USER_EMAIL}/genai-eval-tutorial"
mlflow.set_experiment(EXPERIMENT_PATH)

print(f"MLflow experiment set to: {EXPERIMENT_PATH}")


2026/04/26 08:27:23 INFO mlflow.tracking.fluent: Experiment with name '/Users/sourav.banerjee@databricks.com/genai-eval-tutorial' does not exist. Creating a new experiment.
MLflow experiment set to: /Users/sourav.banerjee@databricks.com/genai-eval-tutorial

---
## Step 4 — Verify Foundation Model API Access

`DatabricksOpenAI` is an OpenAI-compatible client that uses your workspace credentials automatically — no API key to manage. We use it to call `databricks-claude-opus-4-6`, a Foundation Model API endpoint that ships with every Databricks workspace.

If this call fails, check:
- Foundation Model APIs are enabled in your workspace
- The cluster has network egress
- You have CAN_QUERY entitlement on the serving endpoint


In [6]:
# ============================================================================
# 🤖 STEP 4 - VERIFY FOUNDATION MODEL API ACCESS
# ============================================================================

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
client = w.serving_endpoints.get_open_ai_client()

resp = client.chat.completions.create(
    model="databricks-claude-opus-4-6",
    messages=[
        {"role": "user", "content": "What is Delta Lake in one sentence?"}
    ],
)

print(resp.choices[0].message.content)


/root/.ipykernel/12979/command--1-2487479449:8: DeprecationWarning: get_open_ai_client() is deprecated. Please install the databricks-openai package and use 'from databricks_openai import DatabricksOpenAI' instead. See https://api-docs.databricks.com/python/databricks-ai-bridge/latest/databricks_openai.html for more information.
  client = w.serving_endpoints.get_open_ai_client()
Delta Lake is an open-source storage layer that brings ACID transactions, scalable metadata handling, and unified streaming and batch data processing to data lakes, typically built on top of Apache Spark and Parquet files.

---
## Step 5 — Sanity-Check Trace with `mlflow.openai.autolog()`

Autologging hooks every OpenAI-compatible call and emits an **MLflow Trace** — the unit of observability you'll inspect, score, and evaluate in every later module. Run the cell, then open the MLflow experiment to confirm the trace appears under the **Traces** tab.


In [7]:
# ============================================================================
# 🔭 STEP 5 - SANITY-CHECK TRACE WITH `MLFLOW.OPENAI.AUTOLOG()`
# ============================================================================

import mlflow

mlflow.openai.autolog()

resp = client.chat.completions.create(
    model="databricks-claude-opus-4-6",
    messages=[
        {"role": "system", "content": "You are a concise Databricks expert."},
        {"role": "user",   "content": "Name three benefits of MLflow Tracing for GenAI apps."},
    ],
)

print(resp.choices[0].message.content)


Here are three key benefits of MLflow Tracing for GenAI applications:

1. **Observability and Debugging** – MLflow Tracing captures the full execution path of complex GenAI workflows (e.g., chains, agents, RAG pipelines), making it easy to pinpoint where issues like hallucinations, latency bottlenecks, or incorrect retrievals occur.

2. **Quality Evaluation and Improvement** – Traces provide the detailed input/output data at each step needed to systematically evaluate and improve model quality, enabling you to assess retrieval relevance, prompt effectiveness, and overall response accuracy.

3. **Production Monitoring and Cost Tracking** – Tracing records metadata such as token usage, latency, and model parameters for each span, enabling you to monitor production performance, track costs, and identify regressions over time.

### Verify the trace in the UI

1. Click the **Experiments** icon in the left nav
2. Open the experiment at the path printed in Step 3
3. Switch to the **Traces** tab — you should see one trace with the model name, latency, and token counts

You can also fetch the most recent trace programmatically:


In [9]:
# ============================================================================
# 🔭 VERIFY THE TRACE IN THE UI
# ============================================================================

experiment = mlflow.get_experiment_by_name(EXPERIMENT_PATH)
traces = mlflow.search_traces(experiment_ids=[experiment.experiment_id], max_results=1)
display(traces)

trace_id trace client_request_id state request_time execution_duration request response trace_metadata tags spans assessments tr-42c9cdf6aa0da8026fc873858bb25850 {"info": {"trace_id": "tr-42c9cdf6aa0da8026fc873858bb25850", "client_request_id": "tr-42c9cdf6aa0da8026fc873858bb25850", "trace_location": {"type": "MLFLOW_EXPERIMENT", "mlflow_experiment": {"experiment_id": "921609533350370"}}, "request_time": "2026-04-26T08:29:35.081Z", "state": "OK", "trace_metadata": {"mlflow.trace_schema.version": "3", "mlflow.trace.tokenUsage": "{\"input_tokens\": 33, \"output_tokens\": 190, \"total_tokens\": 223}", "mlflow.traceInputs": "{\"model\": \"databricks-claude-opus-4-6\", \"messages\": [{\"role\": \"system\", \"content\": \"You are a concise Databricks expert.\"}, {\"role\": \"user\", \"content\": \"Name three benefits of MLflow Tracing for GenAI apps.\"}]}", "mlflow.trace.sizeStats": "{\"total_size_bytes\": 4799, \"num_spans\": 1, \"max\": 2450, \"p25\": 2450, \"p50\": 2450, \"p75\": 2450}", "mlflow.source.type": "NOTEBOOK", "mlflow.source.git.branch": "", "mlflow.databricks.notebook.commandID": "1001777185416131_6559782183006525135_5cb26cf6023d85148495c98221c44e2a", "mlflow.source.git.repoURL": "", "mlflow.user": "root", "mlflow.source.name": "/databricks/python_shell/scripts/db_ipykernel_launcher.py", "mlflow.traceOutputs": "{\"id\": \"msg_bdrk_01SphnZj3EKCg1aqi6vnnche\", \"choices\": [{\"finish_reason\": \"stop\", \"index\": 0, \"logprobs\": null, \"message\": {\"content\": \"Here are three key benefits of MLflow Tracing for GenAI applications:\\n\\n1. **Observability and Debugging** \u2013 M...", "mlflow.trace.sizeBytes": "4799", "mlflow.source.git.commit": ""}, "tags": {"mlflow.artifactLocation": "dbfs:/databricks/mlflow-tracking/921609533350370/tr-42c9cdf6aa0da8026fc873858bb25850/artifacts", "mlflow.user": "sourav.banerjee@databricks.com", "mlflow.traceName": "Completions"}, "request_preview": "Name three benefits of MLflow Tracing for GenAI apps.", "response_preview": "Here are three key benefits of MLflow Tracing for GenAI applications:\n\n1. **Observability and Debugging** \u2013 MLflow Tracing captures the full execution path of complex GenAI workflows (e.g., chains, agents, RAG pipelines), making it easy to pinpoint where issues like hallucinations, latency bottlenecks, or incorrect retrievals occur.\n\n2. **Quality Evaluation and Improvement** \u2013 Traces provide the detailed input/output data at each step needed to systematically evaluate and improve model quality, enabling you to assess retrieval relevance, prompt effectiveness, and overall response accuracy.\n\n3. **Production Monitoring and Cost Tracking** \u2013 Tracing records metadata such as token usage, latency, and model parameters for each span, enabling you to monitor production performance, track costs, and identify regressions over time.", "execution_duration_ms": 5564}, "data": {"spans": [{"trace_id": "QsnN9qoNqAJvyHOFi7JYUA==", "span_id": "YcMHiqZu4XU=", "parent_span_id": null, "name": "Completions", "start_time_unix_nano": 1777192175081650641, "end_time_unix_nano": 1777192180646166478, "events": [], "status": {"code": "STATUS_CODE_OK", "message": ""}, "attributes": {"mlflow.traceRequestId": "\"tr-42c9cdf6aa0da8026fc873858bb25850\"", "mlflow.spanType": "\"CHAT_MODEL\"", "mlflow.spanInputs": "{\"model\": \"databricks-claude-opus-4-6\", \"messages\": [{\"role\": \"system\", \"content\": \"You are a concise Databricks expert.\"}, {\"role\": \"user\", \"content\": \"Name three benefits of MLflow Tracing for GenAI apps.\"}]}", "model": "\"databricks-claude-opus-4-6\"", "mlflow.message.format": "\"openai\"", "mlflow.llm.model": "\"databricks-claude-opus-4-6\"", "mlflow.chat.tokenUsage": "{\"input_tokens\": 33, \"output_tokens\": 190, \"total_tokens\": 223}", "mlflow.spanOutputs": "{\"id\": \"msg_bdrk_01SphnZj3EKCg1aqi6vnnche\", \"choices\": [{\"finish_reason\": \"stop\", \"index\": 0, \"logprobs\": null, \"message\": {\"content\": \"Here are three key benefits 

---
## Lab Complete — Module 1 Outcome Coverage

> **Outcome:** *You understand the full MLflow 3 evaluation landscape and have a working Databricks environment — experiment, UC schema, Foundation Model access — ready for all labs ahead.*

| Outcome Component | Where It Was Covered | Status |
| --- | --- | --- |
| **MLflow 3 evaluation landscape** | Landscape section above (3 paradigms, module map, why `mlflow.genai`) | ✅ |
| **Working experiment** | Step 3 — `/Users/<you>/genai-eval-tutorial` registered | ✅ |
| **UC schema** | Step 2 — `genai_eval_tutorial.module_01` catalog + schema + volume | ✅ |
| **Foundation Model API access** | Step 4 — verified `databricks-claude-opus-4-6` call | ✅ |
| **Tracing flowing end-to-end** | Step 5 — `mlflow.openai.autolog()` trace visible in UI | ✅ |
| **Packages pinned** | Step 1 — `mlflow[databricks]>=3.1` + `databricks-openai` | ✅ |

Once every row is green, proceed to **Module 2 — Building Evaluation Datasets**.


---
## 📝 Summary

In this notebook, we covered:

### 1. MLflow 3 Evaluation Landscape
- **Three paradigms layered together:** code scorers (deterministic rules), LLM judges (semantic quality at scale), human review (ground truth).
- **`mlflow.genai` is the unified namespace** in MLflow 3 — supersedes the fragmented `mlflow.evaluate` / `mlflow.metrics.genai` APIs from older versions.
- **Module map:** Tracing (M1, M8) → Datasets (M2) → predict_fn patterns (M3) → Judges built-in (M4) and custom (M5) → Human review (M6) → CI/CD (M7) → Production monitoring (M8).

### 2. Environment Bootstrap
- **`mlflow[databricks]>=3.1`** is the floor for GenAI evaluation — older versions lack `mlflow.genai`.
- **`dbutils.library.restartPython()`** is required after `%pip install` so new packages load into the running kernel.

### 3. Unity Catalog Layout
- **Catalog → Schema → Volume** is the canonical place for tutorial assets — works for tables, models, and unstructured files alike.
- Naming the namespace per tutorial (`genai_eval_tutorial.module_01`) keeps later cleanup trivial.

### 4. Experiment Path
- Pinning the experiment under `/Users/<you>/...` makes it visible in the MLflow UI sidebar and isolates runs per learner.

### 5. Tracing Verification
- **`mlflow.openai.autolog()`** is the single line that turns OpenAI-compatible calls into MLflow traces.
- If `search_traces` returns rows, the tracing pipeline is live — every later module depends on this.

### 6. Module 1 Outcome — Coverage Confirmed
- **Outcome:** *You understand the full MLflow 3 evaluation landscape and have a working Databricks environment — experiment, UC schema, Foundation Model access — ready for all labs ahead.*
- **Landscape understanding:** covered by the landscape section + module map.
- **Working environment:** experiment ✅, UC schema ✅, FM API ✅, tracing ✅.

### Next Steps
- **Module 2 — Building Evaluation Datasets** — three ways to create the data your judges will score against.
